## New notebook to read output of MD simulation, using Adios2 library

### part 1: **print information summary**

- Reading one of the sample output files (.bp), step by step.  
- Printing available variables and attributes, as well as their structure.

In [ ]:
import numpy as np
from adios2 import Stream
import os

def read_adios_output(outputDir, index, print_summary=True):
    attributes = {}
    variables = {}
    
    filename = os.path.join(outputDir, f"run_{index}.bp")

    with Stream(filename, "r") as s:
        for i, _ in enumerate(s.steps()):
            if i == 0:
                for attr in s.available_attributes():
                    attributes[attr] = s.read_attribute(attr)
            for var in s.available_variables():
                if var not in variables:
                    variables[var] = []
                variables[var].append(s.read(var))

    for var in variables:
        variables[var] = np.array(variables[var])

    if print_summary:
        print("Attributes Summary:")
        print("-------------------")
        for idx, (name, value) in enumerate(attributes.items(), start=1):
            print(f"{idx}. {name:<40} {value}")
        print("\nVariables Summary:")
        print("------------------")
        for idx, (name, arr) in enumerate(variables.items(), start=1):
            shape = arr.shape
            if arr.ndim == 1:
                description = shape[0]
            else:
                description = f"{shape[0]} * {list(shape[1:])}"
            print(f"{idx}. {name:<35} {description}")

    return {"attributes": attributes, "variables": variables}

output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb13_phaseOr_LJphi2_fmod/outputs/"
result = read_adios_output(output_dir, 0)


---

### Part 2: **Read and Plot Neighbor Counts**

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader

def read_variable(output_dir, variable_name):

    pattern = re.compile("run_([0-9]+).bp")
    run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
    
    if not run_numbers:
        raise ValueError(f"No run files found in {output_dir}")
    
    run_numbers.sort()
    variable_data = []
    temperatures = []
    
    for i, run_num in enumerate(run_numbers):
        file_path = os.path.join(output_dir, f"run_{run_num}.bp")
        with FileReader(file_path) as reader:
            if variable_name in reader.available_variables():
                var_info = reader.available_variables()[variable_name]
                steps = int(var_info.get("AvailableStepsCount", 1))
    
                data = reader.read(variable_name, step_selection=[0, steps])
                if (data.ndim)== 1:
                    data = np.reshape(data, (1*steps, data.shape[0]//steps))
                else:
                    data = np.reshape(data, (1*steps, data.shape[0]//steps, data.shape[1]))

                variable_data.extend(data)
                
                temp_label = reader.read_attribute("temperature")
                temp_label = temp_label.flatten()
                temperatures.append(temp_label)
                
            else:
                raise ValueError(f"Variable {variable_name} not found in {file_path}")
            
    return np.array(variable_data), np.array(temperatures)

def plot_neighbors(neighbors_array, m_temperatures):
    neighbors_data = neighbors_array[0]
    temp_labels = neighbors_array[1]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.viridis(np.linspace(0, 1, neighbors_data.shape[2]+1))
    
    for i in range(neighbors_data.shape[2]):
        y = neighbors_data[:, :, i]
        axs[0].plot(y, label=f"Shell {i}", color=colors[i], linewidth=2)
    
    total_points = neighbors_data.shape[0]
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels) - 1, 10, dtype=int)

    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    
    axs[0].set_ylabel("Number of Neighbors")
    axs[0].set_title("Neighbor Analysis")
    
    axis_labels = ['x', 'y', r'$\omega$']
    
    colors = plt.cm.plasma(np.linspace(0, 1, temperature_data.shape[2]+1))

    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i], linewidth=2)
        
    total_points = temperature_data.shape[0]
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels) - 1, 10, dtype=int)    
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)

    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    plt.show()

def plot_energies(kinetic_energy, potential_energy, m_temperatures, show_potential=True):
    kinetic_energy_data = kinetic_energy[0]
    potential_energy_data = potential_energy[0]/ 100
    temp_labels = kinetic_energy[1]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, kinetic_energy_data.shape[2]+3))
    
    axis_labels = ['x', 'y', r'$\omega$']
    
    for i, axis_label in enumerate(axis_labels):
        y = kinetic_energy_data[:, :, i]
        print("kin", y.shape)
        axs[0].plot(y, label=f"K{axis_label}", color=colors[i])
    if show_potential:    
        y2 = potential_energy_data.flatten()
        axs[0].plot(y2, label='U', color=colors[3])
        axs[0].plot(y2 + np.sum(kinetic_energy_data, axis=2)[:,0], label='Total', color=colors[4])

    total_points = kinetic_energy_data.shape[0]
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels) - 1, 10, dtype=int)

    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    
    axs[0].set_ylabel("Energy")
    axs[0].set_title("Energy Evolution")
     
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])

    
    total_points = temperature_data.shape[0]
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels) - 1, 10, dtype=int)
    
    axs[1].set_xlabel("Temperature Labels")

    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()

def plot_com_velocity(com_velocity, m_temperatures):
    com_vel_data = com_velocity[0]
    temp_labels = com_velocity[1]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, 4))
    
    axis_labels = ['x', 'y']
    # axis_labels = ['x', 'y', r'$\omega$']  #later edit COM calculation
    
    for i, axis_label in enumerate(axis_labels):
        y = com_vel_data[:, :, i]
        axs[0].plot(y, label=f"K{axis_label}", color=colors[i])
    
    total_points = com_vel_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    
    axs[0].set_ylabel("Velocity")
    axs[0].set_title("Center of Mass Velocity Evolution")
     
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])
    
    total_points = temperature_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()

def plot_order_parameter(order, m_temperatures, type):
    order_data = order[0]
    temp_labels = order[1]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, 4))
    
    y = order_data[:, :]
    axs[0].plot(y)
    
    total_points = order_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    
    axs[0].set_ylabel("order parameter")
    axs[0].set_title(f"{type} order parameter")
     
    axis_labels = ['x', 'y'] 
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])
    
    total_points = temperature_data.shape[0]
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    label_indices = np.linspace(0, len(temp_labels)-1, 10, dtype=int)
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in label_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()
    
def plot_trajectory(positions, t_min, t_window):
    positions = positions[0]
    t_max = t_min + t_window
    print("Size of position file = ", positions.shape)
    
    fig, ax = plt.subplots(1, 2, figsize=(8, 4), dpi=150)

    for i in range(positions.shape[1]):
        ax[0].plot(positions[t_min:t_max, i, 0], positions[t_min:t_max, i, 1], 'o', ms=1, alpha=0.6)

    ax[0].set_title("Particle Trajectories")
    ax[0].set_xlabel("X Position")
    ax[0].set_ylabel("Y Position")
    ax[0].grid(linestyle='--', alpha=0.5)

    ax[1].scatter(positions[t_min, :, 0], positions[t_min, :, 1], marker='o', color='blue', label='Initial Position')
    ax[1].scatter(positions[t_max, :, 0], positions[t_max, :, 1], marker='x', color='red', label='Final Position')

    ax[1].set_title("Initial vs. Final Positions")
    ax[1].set_xlabel("X Position")
    ax[1].set_ylabel("Y Position")
    ax[1].legend()
    ax[1].grid(linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

def plot_snapshot_oriented(positions, target_temperature, color_palette):
    positions_data = positions[0]
    temperature_labels = positions[1].flatten()
    print(f"available temperatures are from {min(temperature_labels)} to {max(temperature_labels)}.")
          
    try:
        index = np.where(temperature_labels == target_temperature)[0][0]
    except IndexError:
        print(f"Target temperature {target_temperature} not found in temperature_labels.")
        return

    total_snapshots = positions_data.shape[0]
    shot = int(total_snapshots / len(temperature_labels) * index)

    fig, ax = plt.subplots(figsize=(6, 4))
    
    scatter = ax.scatter(
        positions_data[shot, :, 0],    # X
        positions_data[shot, :, 1],    # Y
        c=positions_data[shot, :, 2],  # φ
        s=50,
        alpha=0.8,
        vmin=-np.pi,
        vmax=np.pi,
        cmap=color_palette
    )
        
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('φ in radian')
        
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 20)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')
    ax.set_title(f'Positions + Orientation φ (T={target_temperature})')
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

def plot_hist_phi(positions, target_temperature, range_read=1000, bins_num=100):
    positions_data = positions[0]
    temperature_labels = positions[1].flatten()
    print(f"available temperatures are from {min(temperature_labels)} to {max(temperature_labels)}.")
    
    try:
        index = np.where(temperature_labels == target_temperature)[0][0]
    except IndexError:
        print(f"Target temperature {target_temperature} not found in temperature_labels.")
        return

    total_snapshots = positions_data.shape[0]
    shot = int(total_snapshots / len(temperature_labels) * index)

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.hist(positions_data[shot-range_read:shot+range_read,:,2].flatten(), bins=bins_num)
    ax.set_title(f'histogram of particles $\phi$ over {range_read} steps, at T = {target_temperature}')
    ax.set_xlabel('orientation angle ($\phi$)')
    ax.set_ylabel('number of particles')
    ax.set_axisbelow(True)
    ax.grid(color='gray', linestyle='dashed')
    
    plt.tight_layout()
    plt.show()
    

In [ ]:
positions = read_variable(output_dir, 'positions')
plot_hist_phi(positions, 0.02)


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm


def map_to_frame(x, y, frame_size, offset=0):
    scale = frame_size / 20
    return int(x * scale + offset), int(y * scale + offset)

def phi_to_color(phi):
    norm_phi = phi / (2 * np.pi) 
    color = cm.twilight(norm_phi)
    bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
    return bgr_color

def temperature_to_color(temp):
    norm_temp = (temp - 0.1) / (1 - 0.1)
    color = cm.coolwarm(norm_temp)
    bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
    return bgr_color
    
def animation_particles(file, particle_type, skip_rows, output_name, phase_transition = False) :
    frame_size = 800
    particle_radius = 10

    type_size = 2 if particle_type == "dot" else 3

    positions = []
    with open(file, 'r') as file:
        for i, line in enumerate(file):
            if i % skip_rows == 0:
                data = line.strip().split()
                data = list(map(float, data[1:]))
                positions.append(data)
    positions = np.array(positions)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_name, fourcc, 10.0, (frame_size + 200, frame_size))

    high_temp = 1.0
    low_temp = 0.1
    num_frames = len(positions)
    temperatures = np.linspace(high_temp, low_temp, num_frames // 2).tolist() + np.linspace(low_temp, high_temp, num_frames - num_frames // 2).tolist()

    for i, frame_data in enumerate(positions):
        frame = np.ones((frame_size, frame_size + 200, 3), dtype=np.uint8) * 255

        # Draw bounding box around particles
        cv2.rectangle(frame, (0, 0), (frame_size, frame_size), (0, 0, 0), 2)

        # Draw particles
        for j in range(0, len(frame_data), type_size):
            x, y = frame_data[j], frame_data[j + 1]
            cx, cy = map_to_frame(x, y, frame_size)

            if type_size == 3:
                phi = frame_data[j + 2] + np.pi
                color = phi_to_color(phi)
            else:
                color = (120, 120, 120)

            cv2.circle(frame, (cx, cy), particle_radius, color, -1)
        if phase_transition == True :
            # Draw temperature bar with gradient
            bar_x_start = frame_size + 50
            bar_x_end = frame_size + 150
            bar_y_start = 50
            bar_y_end = frame_size - 50
            for y in range(bar_y_start, bar_y_end):
                t = (y - bar_y_start) / (bar_y_end - bar_y_start)
                color = cm.coolwarm(1 - t)
                bgr_color = (int(color[2] * 255), int(color[1] * 255), int(color[0] * 255))
                cv2.line(frame, (bar_x_start, y), (bar_x_end, y), bgr_color, 1)

            # Draw indicator for current temperature
            temp = temperatures[i]
            indicator_y = int(bar_y_end - (temp - 0.1) / (1 - 0.1) * (bar_y_end - bar_y_start))
            cv2.arrowedLine(frame, (bar_x_end + 10, indicator_y), (bar_x_end + 50, indicator_y), (0, 0, 0), 2, tipLength=0.3)

        out.write(frame)

        if i % 50 == 0:
            print(f"Processing frame {i}/{len(positions)}")

    out.release()
    print("Video saved as", output_name)

    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("Particle Movement at Last Frame")
    plt.show()

In [ ]:
number_neighbors = read_variable(output_dir, 'number of neighbors')
temperature = read_variable(output_dir, 'real temperature')

plot_neighbors(number_neighbors, temperature)


In [ ]:
kinetic = read_variable(output_dir, 'kinetic energy')
potential = read_variable(output_dir, 'potential energy')

plot_energies(kinetic, potential, temperature, show_potential=True)


In [ ]:
orientational_order = read_variable(output_dir, "order parameter")
plot_order_parameter(orientational_order, temperature, "rotational")

In [ ]:
orientational_order = read_variable(output_dir, "orientational order")
plot_order_parameter(orientational_order, temperature, "rotational")

In [ ]:
com_velocity = read_variable(output_dir, "center of mass velocity")

plot_com_velocity(com_velocity, temperature)

In [ ]:
positions = read_variable(output_dir, 'positions')

plot_trajectory(positions, positions[0].shape[0]//2, 10000)

In [ ]:
plot_snapshot_oriented(positions, 0.02,'hsv')

In [ ]:
positions[0].shape

NOte: Do to list:

1. delta phi over distances (hist , delta phi, distances)
    - closer than a specific distance, plot deltaphi..
    - 3D plot of deltaphi histogram
2. lower temperature

In [ ]:
coldest = positions[0].shape[0] //2
rangeRead = 1000
plt.hist(positions[0][coldest-rangeRead:coldest+rangeRead,:,2].flatten(), bins=100)
plt.title(f'histogram of particles $\phi$ over {rangeRead} steps, at T = {coldest}')
plt.xlabel('orientation angle ($\phi$)')
plt.ylabel('number of particles')
plt.show()

In [ ]:
plt.hist(positions[0][1000:11000,:,2].flatten(), bins=100)
plt.show()

In [ ]:
f = FileReader(os.path.join(output_dir, "run_49.bp"))

In [ ]:
n = 100
pos = f.read("positions", step_selection=(int(f.available_variables()["positions"]["AvailableStepsCount"])-n, n))
pos = pos.reshape(n, pos.shape[0]//n, pos.shape[1])

In [ ]:
plt.hist(pos[4, :,-1])

In [ ]:
plt.hist(pos[:, :,-1].transpose(1,0))
plt.show()

In [ ]:
plt.plot(pos[:,5:7,-1])
plt.show()

In [ ]:
posnew = pos[-1,:,:]
a = np.argwhere(posnew[:,-1] > 1.5)
b = np.argwhere(np.logical_and(posnew[:,-1] > -1, posnew[:,-1] < 0))
ab = np.concatenate([a,b])[:,0]
a = np.argwhere(posnew[:,-1] < -2)[0,0]
posnew[ab,-1] = posnew[a,-1]

In [ ]:
output_dir

In [ ]:
plt.hist(posnew[:,-1])

In [ ]:
vel = f.read("velocities", step_selection=(int(f.available_variables()["velocities"]["AvailableStepsCount"])-1, 1))

In [ ]:
from adios2 import Stream
with Stream("input.bp", "w") as s:
    s.write("positions", posnew, posnew.shape, (0,0), posnew.shape)
    s.write("velocities", vel, vel.shape, (0,0), vel.shape)



In [ ]:
posnew.shape